In [1]:
# Cell 1: Install histdata package + dependencies
import subprocess
pkgs = ["histdata", "pandas", "numpy", "tqdm", "ta"]
for pkg in pkgs:
    r = subprocess.run(["pip", "install", pkg, "-q"], capture_output=True, text=True)
    print(f"{pkg}: {'OK' if r.returncode == 0 else r.stderr[:100]}")

histdata: OK
pandas: OK
numpy: OK
tqdm: OK
ta: OK


In [3]:
# Cell 2: Check what years are available for XAUUSD
from histdata.api import Platform as P, TimeFrame as TF
from histdata import download_hist_data as dl
import pandas as pd
from pathlib import Path

# Test earliest year available
for year in [2000, 2001, 2002, 2003, 2004, 2005]:
    try:
        dl(year=year, month=1, pair='xauusd', 
           platform=P.GENERIC_ASCII, time_frame=TF.ONE_MINUTE,
           output_directory='/home/rocm-user/nexus/data_store/raw/historical/')
        print(f"{year}: ✅ available")
        break
    except Exception as e:
        print(f"{year}: ❌ {str(e)[:60]}")

2000: ❌ For the current year, please specify month=7 for example.
Fo
2001: ❌ For the current year, please specify month=7 for example.
Fo
2002: ❌ For the current year, please specify month=7 for example.
Fo
2003: ❌ For the current year, please specify month=7 for example.
Fo
2004: ❌ For the current year, please specify month=7 for example.
Fo
2005: ❌ For the current year, please specify month=7 for example.
Fo


In [4]:
# Cell 3: Download ALL available years in parallel
from histdata.api import Platform as P, TimeFrame as TF
from histdata import download_hist_data as dl
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import os

RAW_DIR = '/home/rocm-user/nexus/data_store/raw/historical/'
os.makedirs(RAW_DIR, exist_ok=True)

# Build full job list — 2000 (or earliest) through 2025
YEARS = list(range(2000, 2026))
MONTHS = list(range(1, 13))
jobs = [(y, m) for y in YEARS for m in MONTHS]

def download_month(year, month):
    try:
        dl(year=year, month=month, pair='xauusd',
           platform=P.GENERIC_ASCII, time_frame=TF.ONE_MINUTE,
           output_directory=RAW_DIR)
        return (year, month, "OK")
    except Exception as e:
        return (year, month, f"ERR: {str(e)[:50]}")

print(f"Downloading {len(jobs)} month files...")
results = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(download_month, y, m): (y, m) for y, m in jobs}
    for f in tqdm(as_completed(futures), total=len(jobs)):
        results.append(f.result())

# Summary
ok  = [r for r in results if r[2] == "OK"]
err = [r for r in results if r[2] != "OK"]
print(f"\nDownloaded: {len(ok)} months OK")
print(f"Errors    : {len(err)}")
if err[:5]:
    for e in err[:5]:
        print(f"  {e}")

  0%|          | 0/312 [00:00<?, ?it/s]


Downloaded: 0 months OK
Errors    : 312
  (2014, 10, 'ERR: For the current year, please specify month=7 for e')
  (2014, 9, 'ERR: For the current year, please specify month=7 for e')
  (2007, 8, 'ERR: For the current year, please specify month=7 for e')
  (2014, 8, 'ERR: For the current year, please specify month=7 for e')
  (2007, 7, 'ERR: For the current year, please specify month=7 for e')
